# B2 — Spam Classification: KNN vs SVM

**Laboratory Practice III (410246), Group B**

Dataset reference: https://www.kaggle.com/uciml/sms-spam-collection-dataset

Run cells top-to-bottom. Set `CSV_PATH` to your downloaded CSV file to use real data; `None` runs the included synthetic demonstration. Results on synthetic data are not real-world model performance.

In [ ]:
# Set to a local CSV filename/path after downloading the Kaggle dataset.
# Keep None to run with built-in synthetic demonstration data.
CSV_PATH = None  # Example: 'spam.csv'

## Implementation and evaluation

In [8]:

# B2: Compare KNN and SVM Spam Classification using TF-IDF Features

from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set CSV file path (use None for demonstration data)
CSV_PATH = "/emails.csv"


def main():

    # Step 1: Load dataset
    if CSV_PATH and Path(CSV_PATH).exists():
        df = pd.read_csv(CSV_PATH)
        print("Real dataset loaded.")
    else:
        spam = [
            "Claim your cash prize now",
            "You won a free phone",
            "Urgent lottery winner call us",
            "Exclusive discount click here",
            "Get rich fast offer",
            "Free voucher redeem today",
            "Winner selected for reward",
            "Claim bonus money instantly"
        ]

        ham = [
            "Can we meet for lunch",
            "Please send the report",
            "Are you home tonight",
            "Meeting at nine tomorrow",
            "Thanks for the update",
            "Remember to buy groceries",
            "Please review my assignment",
            "See you at the office"
        ]

        df = pd.DataFrame(
            [(s + " " + str(i), "spam") for i in range(8) for s in spam] +
            [(s + " " + str(i), "ham") for i in range(8) for s in ham],
            columns=["text", "label"]
        )

        print("Using synthetic demonstration messages.")

    # Step 2: Rename columns
    df = df.rename(columns={
        "v1": "label",
        "v2": "text",
        "Category": "label",
        "Message": "text"
    })

    # Step 3: Clean dataset
    if not {"label", "text"}.issubset(df.columns):
        raise ValueError("Dataset must contain label and text columns.")

    df = df[["text", "label"]].dropna()

    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df["text"] = df["text"].astype(str)

    df = df[df["label"].isin(["spam", "ham"])]

    if df["label"].nunique() != 2 or df["label"].value_counts().min() < 4:
        raise ValueError("Need at least 4 examples of each class.")

    # Step 4: Split dataset into training and testing
    Xtr, Xte, ytr, yte = train_test_split(
        df["text"],
        df["label"],
        test_size=0.25,
        stratify=df["label"],
        random_state=42
    )

    # Step 5: Apply KNN and SVM
    models = [
        ("KNN", KNeighborsClassifier(n_neighbors=min(5, len(Xtr)))),
        ("SVM", LinearSVC(random_state=42))
    ]

    for name, model in models:

        # Step 6: TF-IDF and classifier pipeline
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(stop_words="english")),
            ("classifier", model)
        ])

        # Step 7: Train model
        pipe.fit(Xtr, ytr)

        # Step 8: Predict results
        pred = pipe.predict(Xte)

        # Step 9: Evaluate model
        print("\n==========", name, "==========")

        print("Accuracy:", round(accuracy_score(yte, pred), 4))

        print("Confusion Matrix [ham, spam]:")
        print(confusion_matrix(yte, pred, labels=["ham", "spam"]))

        print("Classification Report:")
        print(classification_report(
            yte,
            pred,
            labels=["ham", "spam"],
            zero_division=0
        ))


if __name__ == "__main__":
    main()


Using synthetic demonstration messages.

========== KNN ==========
Accuracy: 1.0
Confusion Matrix [ham, spam]:
[[16  0]
 [ 0 16]]
Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00        16
        spam       1.00      1.00      1.00        16

    accuracy                           1.00        32
   macro avg       1.00      1.00      1.00        32
weighted avg       1.00      1.00      1.00        32


========== SVM ==========
Accuracy: 1.0
Confusion Matrix [ham, spam]:
[[16  0]
 [ 0 16]]
Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00        16
        spam       1.00      1.00      1.00        16

    accuracy                           1.00        32
   macro avg       1.00      1.00      1.00        32
weighted avg       1.00      1.00      1.00        32

